# Songs 1970–1980: genres, lyrics and networks

MusicBrainz songs (metadata, genres, credits) matched to Genius lyrics. This notebook shows the key results; the code lives in [src/](src/) (see [README.md](README.md)), and the planned methods are in [Documentation.ipynb](Documentation.ipynb).

1. **Data overview**: what survives cleaning and matching, and do the two sources agree on genre?
2. **Genre network**: songs linked by shared genres, and its communities
3. **Lyrics network**: songs linked by similar lyrics. Do the communities match the genres, and is that more than chance?
4. **Genre classification**: can lyrics predict a song's genre?

Run top to bottom. The processed data is loaded from `data/processed/`, and it's rebuilt from `data/raw/` only if missing or when `REBUILD = True`. Lyrics are tokenized with spaCy once and cached. The lyrics-network analysis takes about 20 minutes, so its saved results in `results/` are shown unless `RERUN_LYRICS_NETWORK = True`.

In [ ]:
REBUILD = False               # True: re-run cleaning + lyrics matching from data/raw/ (a few minutes)
RERUN_LYRICS_NETWORK = False  # True: re-run the lyrics network analysis (~20 min) instead of loading results/

import pandas as pd
from IPython.display import Image, Markdown, display

from src import classification, exploration, genre_network, lyrics_network
from src.config import FIG_DIR, FIRST_YEAR, YEARS
from src.data import load_data

pd.set_option("display.max_colwidth", 120)
%matplotlib inline

songs, genius = load_data(rebuild=REBUILD)
print(f"MusicBrainz songs {YEARS}: {len(songs)}   ({songs.lyrics.notna().sum()} with lyrics)")
print(f"Genius songs {YEARS}: {len(genius)}")

## 1. Data overview

### How many songs survive each step?
**MusicBrainz:** raw recordings, merged into songs. We then count how many have a genre, how many of those genres map onto a Genius tag, how many got lyrics, and how many have labels from both sources that agree.

In [ ]:
exploration.musicbrainz_funnel(songs)

**Genius:** the cleaning steps, plus the subsets used in the network and classification analyses (rap is dropped there, since there are too few songs).

In [ ]:
exploration.genius_funnel(genius)

### How were the lyrics matched?
Exact title + artist first. Looser rules are only tried when that fails.

In [ ]:
songs.match.value_counts().to_frame("songs")

### Do MusicBrainz and Genius agree on the genre?
Rows: the Genius tag. Columns: the MusicBrainz genres mapped onto the same five tags (`mb_label`). Only songs that have both are counted. The diagonal is agreement.

In [ ]:
agree = exploration.label_agreement(songs)
display(agree)
print(f"Agreement: {songs.labels_agree.mean():.0%} of {songs.labels_agree.notna().sum()} songs with both labels")

### Credit coverage: MusicBrainz vs Discogs
How many songs of the first year in `src/config.py` have instrument and producer credits, per genre, and whether credited songs are spread evenly over the genres (*over-rep.* = 1 means proportional). This needs `songs_<year>.csv` crawled with credits, and optionally `discogs_<year>.csv`.

In [ ]:
coverage = exploration.credit_coverage(FIRST_YEAR)
if coverage is None:
    print(f"data/raw/songs_{FIRST_YEAR}.csv has no credit columns yet: re-crawl with python -m src.download.musicbrainz")
else:
    display(coverage.style.format("{:.2f}"))

### The dataset at a glance
Songs per year, the most common primary genres and release countries, and where lyrics were found. The bottom right is the agreement table above, as row percentages.

In [ ]:
exploration.plot_overview(songs, genius);

### The lyrics
Length and chorus repetition per Genius tag. *Distinctive words*: how many times more of the tag's songs use the word than songs of the other tags (counted per song). Rap is left out of that row, since it has too few matched songs.

In [ ]:
lyr = exploration.lyric_stats(songs)
exploration.plot_lyrics(lyr, exploration.distinctive_words(lyr));

**Most common words** across all matched lyrics, after tokenization (spaCy, stop words and section markers removed, words that occur in only one song dropped). From the token branch's tokenization notebook.

In [ ]:
from collections import Counter

in_songs = Counter(w for tokens in lyr.tokens for w in set(tokens))
counts = Counter(w for tokens in lyr.tokens for w in tokens if in_songs[w] >= 2)
pd.DataFrame(counts.most_common(30), columns=["word", "count"]).assign(
    songs=lambda d: d.word.map(in_songs)).set_index("word").T

## 2. Genre network

Each node is a song with at least one MusicBrainz genre. Two songs are linked when their genre lists overlap; edge weight is the number of shared genres. Each genre adds at most 500 randomly sampled links, so rock doesn't swamp the graph. Greedy modularity finds the communities.

To read the numbers: modularity runs from 0 (no community structure) to 1. *Modularity of primary genres* scores the primary genre labels used as communities, for comparison. *Purity* is the share of a community that has its most common primary genre.

In [ ]:
stats, summary, _, _ = genre_network.analyse(songs)
display(stats.round(3).to_frame("genre network"))
display(Markdown("**Communities with at least 20 songs**"))
display(summary.style.format({"purity": "{:.0%}"}))
display(Markdown(f"Interactive view (open in a browser; hover a song to see its links): "
                 f"[figures/genre_network.html]({(FIG_DIR / 'genre_network.html').relative_to(FIG_DIR.parent).as_posix()})"))

## 3. Lyrics similarity network

Each song is a node, linked to the songs with the most similar lyrics: cosine similarity of TF-IDF vectors over the spaCy tokens. Genre is **never** used to build edges; it's only used afterwards, to evaluate. The genre is `mb_label` (rock, rb, pop, country; rap is dropped). Songs by the same artist are never linked, so a community can't just be one artist's catalogue.

Variants:
- **k** = 5, 10 and 20 nearest neighbours, as a sensitivity check
- **mutual**: two songs are linked only if each is among the other's k most similar. This removes the hub songs, generic "baby, I love you" lyrics that are everyone's neighbour.
- **artists**: one node per artist, using the mean of its songs' vectors; the genre is the artist's most common one

How to read the table:
- **same_genre**: share of links joining two songs of the same genre. Compare it with *same_genre_null* (1,000 genre shuffles) and *rewired_same_genre* (10 degree-preserving rewirings). The *z* columns say how many standard deviations above those nulls it is.
- **<genre>_lift**: the share of a genre's links that stay inside the genre, divided by chance. 1 means no lyrical identity; 2 means twice chance.
- **louvain_modularity / nmi**: community structure, and its agreement with genre (NMI 0 = none, 1 = identical). Each is averaged over 10 Louvain seeds; *seed_stability_nmi* shows how much the partitions agree across seeds.
- **rewired_modularity**: the same Louvain run on rewired graphs. Real structure should clearly beat it.

In [ ]:
saved = None if RERUN_LYRICS_NETWORK else lyrics_network.load_results()
if saved is None:
    lyrics_net, lyrics_comms = lyrics_network.run(songs, verbose=False)
else:
    lyrics_net, lyrics_comms = saved
    print("Showing saved results from results/ (set RERUN_LYRICS_NETWORK = True to recompute)")

cols = ["graph", "mutual", "k", "nodes", "edges", "isolated", "same_genre", "same_genre_null", "same_genre_z",
        "rewired_same_genre", "rock_lift", "rb_lift", "pop_lift", "country_lift", "genre_modularity",
        "louvain_modularity", "rewired_modularity", "communities", "nmi", "nmi_z", "seed_stability_nmi"]
lyrics_net[cols].style.format(precision=3).format("{:.0f}", subset=["k", "nodes", "edges", "isolated", "communities"])

**Links that stay inside a genre, relative to chance** (k = 10; the dashed line at 1 means no lyrical identity)

In [ ]:
display(Image(FIG_DIR / "lyrics_genre_lift.png", width=650))

**Genre mix and typical words per community**: mutual song graph, k = 10, best-modularity partition, communities with at least 10 songs (each row sums to 100%)

In [ ]:
lyrics_comms[lyrics_comms.songs >= 10]

The same graph in one layout, coloured by Louvain community (left) and by genre (right); songs without links are left out. If lyrics separated the genres, the two panels would look alike. An interactive version is written to `figures/lyrics_network.html`.

In [ ]:
display(Image(FIG_DIR / "lyrics_network.png", width=1100))

## 4. Genre classification from lyrics

Three predictors, cross-validated with 5 folds. All of an artist's songs stay on the same side of each split (credits like "X and The Y" count as X), so a model can't score by recognising an artist.
- **guess**: random guessing in proportion to how common each genre is (the baseline)
- **counts**: hand-made features: length, vocabulary richness, repeated lines, word length
- **words**: TF-IDF over the spaCy tokens + logistic regression

The main label is `mb_label`, on the songs that have lyrics. The Genius tag, the songs where both sources agree, and all Genius songs from the period are checks. Scores are macro-F1 (1.0 = perfect). Every genre counts equally, so always answering the biggest genre doesn't pay off.

In [ ]:
scores, confusions = classification.run_all(songs, genius, verbose=False)
scores.style.format({"guess": "{:.2f}", "counts": "{:.2f}", "words": "{:.2f}"})

**Where the word model goes wrong**. Rows are the true genre and columns the prediction, as row percentages, so the diagonal is the recall per genre.

In [ ]:
for name, confusion in confusions.items():
    display(Markdown(f"**{name}**"))
    display(confusion.div(confusion.sum(axis=1), axis=0).style.format("{:.0%}").background_gradient(cmap="Blues", vmin=0, vmax=1))